In [1]:
library(CellChat)
library(Seurat)
library(NMF)
library(ggalluvial)
library(ComplexHeatmap)
options(stringsAsFactors = FALSE)
library(dplyr)
library(ggplot2)

library(ggrepel)
library(circlize)

library(RColorBrewer)
library(scales)

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: igraph


Attaching package: ‘igraph’


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union


Loading required package: ggplot2

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:BiocGenerics’:

    intersect


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘Seurat’


The following object is masked from ‘package:igraph’:

    components


Loading required package: registry

Loading required package: rngto

In [2]:
cellphoneDB_dir="/media/xili/fed0ce75-c75a-4b08-8979-104ef6d66780/cellphoneDB/"

In [3]:
interaction_input <- read.csv(file = paste0(cellphoneDB_dir,'interaction_input_CellChatDB.csv'), row.names = 1)
complex_input <- read.csv(file = paste0(cellphoneDB_dir,'interaction_input_CellChatDB.csv'), row.names = 1)
cofactor_input <- read.csv(file = paste0(cellphoneDB_dir,'interaction_input_CellChatDB.csv'), row.names = 1)
geneInfo <- read.csv(file = paste0(cellphoneDB_dir,'interaction_input_CellChatDB.csv'), row.names = 1)
CellChatDB <- list()
CellChatDB$interaction <- interaction_input
CellChatDB$complex <- complex_input
CellChatDB$cofactor <- cofactor_input
CellChatDB$geneInfo <- geneInfo

In [4]:
CellChatDB <- CellChatDB.mouse

In [5]:
project_list=c('M20_3_V4', 'M20_3_V5', 'M20_5_V21',
       'M20_5_V47', 'M20_V40', 'M20_V42')

In [124]:
outputdir="transcriptome/pool_DEG/"

In [8]:
counts=read.table(paste0(outputdir,"pool_raw.csv"),header=T,sep=",",as.is=T,row.names=1)


In [9]:
meta_everything_anno <- read.table(paste0(outputdir,"meta_treatment_anno.csv"), 
                         header=T, sep=",", as.is=T, row.names=1)

In [10]:

meta_everything <- unclass(meta_everything_anno)
meta_everything_factor <- as.factor(meta_everything$anno_sub)

data <- CreateSeuratObject(counts = counts, project = "all")
data<-AddMetaData(data, meta_everything_factor, col.name="anno_sub")

meta_everything_treatment <- as.factor(meta_everything$Treatment_Sample)
data <- AddMetaData(data, meta_everything_treatment, col.name="Treatment_Sample")
data <- NormalizeData(data , scale.factor = 1e6, normalization.method = "LogNormalize")


In [11]:
data@meta.data$anno_sub = factor(data@meta.data$anno_sub, levels =levels(data@meta.data$anno_sub))

In [24]:
sqrt(dim(data_sub)[2])

[1] 100.2198

In [28]:
for (i in project_list) {
project=i


data_sub=subset(x = data, Treatment_Sample==i)

data_sub <- FindVariableFeatures(data_sub, mean.function = ExpMean, dispersion.function = LogVMR,
	                          selection.method = "vst", nfeatures = 20000)
data_sub <- ScaleData(data_sub)
cellchat <- createCellChat(object = data_sub,  group.by = "anno_sub")
#cellchat@meta$anno_clusters = factor(cellchat@meta$anno_clusters, levels =levels(cellchat@idents))
cellchat <- setIdent(cellchat, ident.use = "anno_sub") 
groupSize <- as.numeric(table(cellchat@idents))

#CellChatDB <- CellChatDB.mouse
#showDatabaseCategory(CellChatDB)
CellChatDB.use <- CellChatDB
cellchat@DB <- CellChatDB.use
cellchat <- subsetData(cellchat)

#future::plan("multiprocess", workers = 1)

cellchat <- identifyOverExpressedGenes(cellchat, thresh.pc = 0.1, thresh.fc = 2 ,thresh.p = 0.05)
cellchat <- identifyOverExpressedInteractions(cellchat)
#cellchat <- projectData(cellchat, PPI.human)

cellchat@idents = droplevels(cellchat@idents, exclude = setdiff(levels(cellchat@idents),unique(cellchat@idents)))
cellchat <- computeCommunProb(cellchat, raw.use=T, type="truncatedMean",k.min=int(sqrt(dim(data_sub)[2])),
	                           #trim=0.2, 
                              nboot = 24, population.size = T )
cellchat <- filterCommunication(cellchat, min.cells = sqrt(dim(data_sub)[2])*2)

cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)

groupSize <- as.numeric(table(cellchat@idents))
par(mfrow = c(1,1), xpd=TRUE)

pdf(paste0(outputdir,project, "_","Visual_circle", ".pdf"),height =10, width = 12)
netVisual_circle(cellchat@net$weight, vertex.weight = groupSize, weight.scale = T, label.edge= F, title.name = "Interaction weights/strength")
dev.off()


saveRDS (cellchat, file=paste0(outputdir,project,"_cellchat.rds"))
dev.off()
}


Centering and scaling data matrix



[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  B.APC, B.Bank1, B.Germ, B.Lagls1, B.secret, B.Sell, CD4.exh, CD4.naive, CD4.reg, CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, DC.migrat, Fibro.cells, Gran.cells, ID8, ID8_ADV, ID8_apop, ID8_apop_ADV, ID8_prolif, ID8_prolif_ADV, MDSC, Mo.DC, NK.T, pre.DC, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
The number of highly variable ligand-receptor pairs used for signaling inference is 130 
truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2024-11-21 17:54:51.605417]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2024-11-21 17:55:53.65429]"
The cell-cell c

Centering and scaling data matrix



[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  B.APC, B.Bank1, B.Germ, B.Lagls1, B.secret, B.Sell, CD4.exh, CD4.naive, CD4.reg, CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, DC.migrat, Fibro.cells, Gran.cells, ID8, ID8_ADV, ID8_apop, ID8_apop_ADV, ID8_prolif, ID8_prolif_ADV, MDSC, Mo.DC, NK.T, pre.DC, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
The number of highly variable ligand-receptor pairs used for signaling inference is 139 
truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2024-11-21 17:56:11.348068]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2024-11-21 17:57:10.670937]"
The cell-cell 

Centering and scaling data matrix



[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  B.APC, B.Bank1, B.Germ, B.Lagls1, B.secret, B.Sell, CD4.exh, CD4.naive, CD4.reg, CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, DC.migrat, Fibro.cells, Gran.cells, ID8, ID8_ADV, ID8_apop, ID8_apop_ADV, ID8_prolif, ID8_prolif_ADV, MDSC, Mo.DC, NK.T, pre.DC, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
The number of highly variable ligand-receptor pairs used for signaling inference is 262 
truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2024-11-21 17:57:27.079259]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2024-11-21 17:58:29.018377]"
The cell-cell 

... (2 outputs left out)

Centering and scaling data matrix



[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  B.APC, B.Bank1, B.Germ, B.Lagls1, B.secret, B.Sell, CD4.exh, CD4.naive, CD4.reg, CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, DC.migrat, Fibro.cells, Gran.cells, ID8, ID8_ADV, ID8_apop, ID8_apop_ADV, ID8_prolif, ID8_prolif_ADV, MDSC, Mo.DC, NK.T, pre.DC, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
The number of highly variable ligand-receptor pairs used for signaling inference is 186 
truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2024-11-21 17:59:58.906786]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2024-11-21 18:00:59.604722]"
The cell-cell 

Centering and scaling data matrix



[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  B.APC, B.Bank1, B.Germ, B.Lagls1, B.secret, B.Sell, CD4.exh, CD4.naive, CD4.reg, CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, DC.migrat, Fibro.cells, Gran.cells, ID8, ID8_ADV, ID8_apop, ID8_apop_ADV, ID8_prolif, ID8_prolif_ADV, MDSC, Mo.DC, NK.T, pre.DC, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
The number of highly variable ligand-receptor pairs used for signaling inference is 247 
truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2024-11-21 18:01:17.291601]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2024-11-21 18:02:21.497911]"
The cell-cell 

In [7]:
treatment_names <- c('M20_3_V4', 'M20_3_V5', 'M20_5_V21', 'M20_5_V47', 'M20_V40', 'M20_V42')

# Initialize list to store CellChat objects
cellchats <- list()

# Load all RDS files
for (treatment in treatment_names) {
  # Construct file name (adjust the path and extension as needed)
  filename <- paste0(outputdir,treatment, "_cellchat.rds")  # or whatever your file naming convention is
  
  cat("Loading", filename, "...\n")
  
  # Load the CellChat object
  cellchat_obj <- readRDS(filename)
  
  # Store in the list with treatment name
  cellchats[[treatment]] <- cellchat_obj
}

# Verify loading
cat("Loaded", length(cellchats), "CellChat objects:\n")
print(names(cellchats))


Loading transcriptome/pool_DEG/M20_3_V4_cellchat.rds ...
Loading transcriptome/pool_DEG/M20_3_V5_cellchat.rds ...
Loading transcriptome/pool_DEG/M20_5_V21_cellchat.rds ...
Loading transcriptome/pool_DEG/M20_5_V47_cellchat.rds ...
Loading transcriptome/pool_DEG/M20_V40_cellchat.rds ...
Loading transcriptome/pool_DEG/M20_V42_cellchat.rds ...
Loaded 6 CellChat objects:
[1] "M20_3_V4"  "M20_3_V5"  "M20_5_V21" "M20_5_V47" "M20_V40"   "M20_V42"  


In [8]:
# 定义参考组优先级
reference_groups <- c( "M20_5_V21","M20_5_V47", "M20_3_V4","M20_3_V5")  # 优先级顺序

sample_pairs <- combn(names(cellchats), 2, simplify = FALSE)

sample_pairs <- lapply(sample_pairs, function(pair) {
  # 找到这一对中优先级最高的参考组
  for (ref in reference_groups) {
    if (ref %in% pair) {
      if (pair[1] != ref) {
        return(c(ref, setdiff(pair, ref)))
      }
      break
    }
  }
  return(pair)  # 如果没有参考组，保持原顺序
})

cat("处理顺序（按参考组优先级）:\n")
for (i in 1:length(sample_pairs)) {
  cat(i, ":", sample_pairs[[i]][1], "vs", sample_pairs[[i]][2], "\n")
}

处理顺序（按参考组优先级）:
1 : M20_3_V4 vs M20_3_V5 
2 : M20_5_V21 vs M20_3_V4 
3 : M20_5_V47 vs M20_3_V4 
4 : M20_3_V4 vs M20_V40 
5 : M20_3_V4 vs M20_V42 
6 : M20_5_V21 vs M20_3_V5 
7 : M20_5_V47 vs M20_3_V5 
8 : M20_3_V5 vs M20_V40 
9 : M20_3_V5 vs M20_V42 
10 : M20_5_V21 vs M20_5_V47 
11 : M20_5_V21 vs M20_V40 
12 : M20_5_V21 vs M20_V42 
13 : M20_5_V47 vs M20_V40 
14 : M20_5_V47 vs M20_V42 
15 : M20_V40 vs M20_V42 


In [380]:
# CellChat 2×2 设计分析工具
# 完整重构版本


# ============================================================================
# 1. 数据提取函数
# ============================================================================

extract_communication_matrix <- function(cellchat_obj, sample_name, target_types) {
  
  # 检查可用细胞类型
  available_celltypes <- intersect(rownames(cellchat_obj@net$weight), target_types)
  
  # 打印信息
  cat("样本", sample_name, "中找到的细胞亚群:", 
      length(available_celltypes), "/", length(target_types), "\n")
  
  if (length(available_celltypes) > 0) {
    cat("具体亚群:", paste(available_celltypes, collapse = ", "), "\n")
  }
  
  # 创建完整矩阵（包含所有目标类型）
  full_mat <- matrix(0, 
                     nrow = length(target_types), 
                     ncol = length(target_types),
                     dimnames = list(target_types, target_types))
  
  # 填充可用数据
  if (length(available_celltypes) > 0) {
    sub_mat <- cellchat_obj@net$weight[available_celltypes, available_celltypes]
    full_mat[available_celltypes, available_celltypes] <- as.matrix(sub_mat)
  }
  
  # 转换为数据框
  df <- as.data.frame(as.table(full_mat))
  colnames(df) <- c("source", "target", "strength")
  df$sample <- sample_name
  df$available <- ifelse(df$source %in% available_celltypes & 
                         df$target %in% available_celltypes, "Yes", "No")
  
  return(df)
}

# ============================================================================
# 2. 样本比较函数
# ============================================================================

compare_two_samples <- function(df, sample1, sample2) {
  
  # 筛选两个样本的数据
  data1 <- df %>% filter(sample == sample1)
  data2 <- df %>% filter(sample == sample2)
  
  # 合并数据
  merged <- merge(data1, data2, 
                  by = c("source", "target"), 
                  suffixes = c(paste0("_", sample1), paste0("_", sample2)))
  
  # 计算差异
  strength_col1 <- paste0("strength_", sample1)
  strength_col2 <- paste0("strength_", sample2)
  
  merged$strength_diff <- merged[[strength_col1]] - merged[[strength_col2]]
  
  # 计算倍数变化（注意处理0值）
  merged$fold_change <- ifelse(merged[[strength_col1]] > 0,
                               merged[[strength_col2]] / merged[[strength_col1]],
                               NA)
  
  # 判断变化方向
  merged$change_direction <- case_when(
    merged$strength_diff > 0 ~ "upregulated",
    merged$strength_diff < 0 ~ "downregulated", 
    TRUE ~ "unchanged"
  )
  
  # 仅保留在两个样本中都可用的交互
  avail_col1 <- paste0("available_", sample1)
  avail_col2 <- paste0("available_", sample2)
  
  merged <- merged %>% 
    filter(get(avail_col1) == "Yes" & get(avail_col2) == "Yes")
  
  return(merged)
}

# ============================================================================
# 3. 小样本统计分析函数（修复版本）
# ============================================================================

#' 小样本量统计分析（2×2设计） - 修复版本
perform_statistical_analysis_2x2 <- function(all_pairwise_results, 
                                             group_A_samples, 
                                             group_B_samples) {
  
  # 验证样本数量
  expected_pairs <- length(group_A_samples) * length(group_B_samples)
  actual_pairs <- length(unique(all_pairwise_results$pair_id))
  
  cat("统计设计: 每组2个样本，共有", actual_pairs, "个比较对\n")
  cat("注意: 统计功效有限，建议谨慎解释结果\n")
  
  # 第一步：计算基本统计
  statistical_results <- all_pairwise_results %>%
    group_by(source, target) %>%
    summarize(
      # 基本描述统计
      n_pairs = n(),
      mean_diff = mean(strength_diff, na.rm = TRUE),
      median_diff = median(strength_diff, na.rm = TRUE),
      sd_diff = sd(strength_diff, na.rm = TRUE),
      se_diff = ifelse(n_pairs > 1, sd_diff / sqrt(n_pairs), NA),
      
      # 方向一致性
      n_up = sum(change_direction == "upregulated"),
      n_down = sum(change_direction == "downregulated"),
      n_unchanged = sum(change_direction == "unchanged"),
      prop_same_direction = max(n_up, n_down) / n_pairs,
      dominant_direction = case_when(
        n_up == n_pairs ~ "all_up",
        n_down == n_pairs ~ "all_down",
        n_up > n_down ~ "mostly_up",
        n_down > n_up ~ "mostly_down",
        TRUE ~ "inconsistent"
      ),
      
      # 倍数变化
      median_fc = median(fold_change, na.rm = TRUE),
      mean_fc = mean(fold_change, na.rm = TRUE),
      
      .groups = "drop"
    )
  
  # 第二步：对每一行单独计算统计检验
  add_statistical_tests <- function(stat_df, pairwise_df) {
    results <- list()
    
    for (i in 1:nrow(stat_df)) {
      row <- stat_df[i, ]
      
      # 提取差异值
      diff_values <- pairwise_df %>%
        filter(source == row$source, target == row$target) %>%
        pull(strength_diff)
      
      # 初始化统计结果
      stats <- list(
        t_test_p_value = NA_real_,
        t_statistic = NA_real_,
        wilcox_p_value = NA_real_,
        sign_test_p_value = NA_real_,
        n_positive = sum(diff_values > 0, na.rm = TRUE),
        n_negative = sum(diff_values < 0, na.rm = TRUE)
      )
      
      # t检验
      if (row$n_pairs >= 2 && row$sd_diff > 0 && length(diff_values) >= 2) {
        tryCatch({
          t_result <- t.test(diff_values, mu = 0)
          stats$t_test_p_value <- t_result$p.value
          stats$t_statistic <- t_result$statistic
        }, error = function(e) {})
      }
      
      # Wilcoxon检验
      if (row$n_pairs >= 2 && length(diff_values) >= 2) {
        tryCatch({
          wilcox_result <- wilcox.test(diff_values, mu = 0, exact = FALSE)
          stats$wilcox_p_value <- wilcox_result$p.value
        }, error = function(e) {})
      }
      
      # 符号检验
      if (row$n_pairs >= 1) {
        tryCatch({
          sign_test <- binom.test(max(stats$n_positive, stats$n_negative), 
                                 row$n_pairs, p = 0.5, alternative = "two.sided")
          stats$sign_test_p_value <- sign_test$p.value
        }, error = function(e) {})
      }
      
      results[[i]] <- as.data.frame(stats)
    }
    
    # 合并结果
    tests_df <- bind_rows(results)
    return(cbind(stat_df, tests_df))
  }
  
  # 添加统计检验
  statistical_results <- add_statistical_tests(statistical_results, all_pairwise_results)
  
  # 第三步：计算衍生统计
  statistical_results <- statistical_results %>%
    mutate(
      # 选择p值（优先Wilcoxon）
      p_value_raw = case_when(
          !is.na(t_test_p_value) ~ t_test_p_value,
        !is.na(wilcox_p_value) ~ wilcox_p_value,
        
        !is.na(sign_test_p_value) ~ sign_test_p_value,
        TRUE ~ NA_real_
      ),
      
      # 效应大小
      cohens_d = ifelse(sd_diff > 0, mean_diff / sd_diff, NA_real_),
      
      # 一致性评分
      consistency_score = case_when(
        prop_same_direction == 1 ~ 4,
        prop_same_direction >= 0.75 ~ 3,
        prop_same_direction >= 0.5 ~ 2,
        TRUE ~ 1
      ),
      
      # 变异系数
      cv = ifelse(mean_diff != 0, sd_diff / abs(mean_diff), NA_real_)
    )
  
  # 第四步：多重检验校正 - 修复p.adjust的使用
  # 创建p值的副本，用于校正
  p_values <- statistical_results$p_value_raw
  
  if (sum(!is.na(p_values)) > 1) {
    # 只对非NA的p值进行校正
    p_adjusted <- rep(NA_real_, length(p_values))
    non_na_indices <- which(!is.na(p_values))
    
    if (length(non_na_indices) > 1) {
      p_adjusted[non_na_indices] <- p.adjust(p_values[non_na_indices], method = "fdr")
    } else if (length(non_na_indices) == 1) {
      # 只有一个p值时，不能进行多重检验校正
      p_adjusted[non_na_indices] <- p_values[non_na_indices]
    }
    
    statistical_results$p_adjusted <- p_adjusted
  } else {
    statistical_results$p_adjusted <- statistical_results$p_adjusted
  }
  
  # 第五步：标注显著性
  statistical_results <- statistical_results %>%
    mutate(
      significance = case_when(
        p_adjusted < 0.05 ~ "**",
        p_adjusted < 0.1 ~ "*",
        p_adjusted < 0.2 ~ ".",
        TRUE ~ "ns"
      ),
      
      priority_level = case_when(
        consistency_score == 4 & abs(mean_diff) > 0.2 ~ "high",
        consistency_score >= 3 & abs(mean_diff) > 0.15 ~ "medium_high",
        consistency_score >= 2 & abs(mean_diff) > 0.1 ~ "medium",
        TRUE ~ "low"
      )
    )
  
  # 打印摘要
  cat("\n统计分析摘要（2×2设计）:\n")
  cat("总交互作用数量:", nrow(statistical_results), "\n")
  cat("高度一致(4/4同向):", sum(statistical_results$consistency_score == 4), "\n")
  cat("中等一致(3/4同向):", sum(statistical_results$consistency_score == 3), "\n")
  cat("边缘一致(2/4同向):", sum(statistical_results$consistency_score == 2), "\n")
  cat("显著(p<0.05):", sum(statistical_results$p_adjusted < 0.05, na.rm = TRUE), "\n")
  cat("边缘显著(p<0.1):", sum(statistical_results$p_adjusted < 0.1, na.rm = TRUE), "\n")
  
  return(statistical_results)
}

# ============================================================================
# 4. 网络可视化函数
# ============================================================================

plot_cell_network_2x2 <- function(statistical_results, comp_name, outputdir) {
  
  # 筛选重要交互
  network_data <- statistical_results %>%
    filter(priority_level %in% c("high", "medium_high", "medium")) %>%
    mutate(
      edge_weight = consistency_score * abs(mean_diff),
      edge_color = case_when(
        mean_diff > 0 ~ "#E76F51",   # A>B (上调)
        mean_diff < 0 ~ "#2A9D8F",   # A<B (下调)
        TRUE ~ "gray"
      ),
      edge_width = case_when(
        priority_level == "high" ~ 4,
        priority_level == "medium_high" ~ 3,
        priority_level == "medium" ~ 2,
        TRUE ~ 1
      ),
      edge_label = paste0(
        "一致性:", consistency_score, "/4\n",
        "Δ=", round(mean_diff, 3), "\n",
        ifelse(!is.na(p_adjusted), 
               paste0("p=", round(p_adjusted, 3)), 
               "p=NA")
      )
    )
  
  # 检查是否有足够数据
  if (nrow(network_data) == 0) {
    cat("警告：没有足够一致的变化，无法绘制网络图\n")
    cat("考虑降低筛选阈值或检查数据\n")
    return(NULL)
  }
  
  # 计算节点重要性
  node_importance <- bind_rows(
    network_data %>% 
      group_by(source) %>%
      summarise(
        outgoing_strength = sum(consistency_score * abs(mean_diff), na.rm = TRUE),
        outgoing_n = n(),
        .groups = "drop"
      ) %>%
      rename(cell = source),
    network_data %>% 
      group_by(target) %>%
      summarise(
        incoming_strength = sum(consistency_score * abs(mean_diff), na.rm = TRUE),
        incoming_n = n(),
        .groups = "drop"
      ) %>%
      rename(cell = target)
  ) %>%
    group_by(cell) %>%
    summarise(
      total_strength = sum(outgoing_strength, incoming_strength, na.rm = TRUE),
      total_connections = sum(outgoing_n, incoming_n, na.rm = TRUE),
      .groups = "drop"
    )
  
  # 创建igraph对象
  g <- graph_from_data_frame(
    d = network_data[, c("source", "target", "edge_weight", 
                         "edge_color", "edge_width", "edge_label")],
    vertices = node_importance,
    directed = TRUE
  )
  
  # 设置节点属性
  V(g)$size <- scales::rescale(V(g)$total_strength, to = c(8, 25))
  V(g)$label.cex <- scales::rescale(V(g)$total_strength, to = c(0.6, 1.0))
  V(g)$color <- "lightblue"
  V(g)$frame.color <- "darkblue"
  
  # 设置边属性
  E(g)$color <- E(g)$edge_color
  E(g)$width <- E(g)$edge_width
  E(g)$arrow.size <- 0.6
  E(g)$arrow.width <- 1.0
  E(g)$label <- E(g)$edge_label
  E(g)$label.color <- "black"
  E(g)$label.cex <- 0.5
  
  # 创建输出目录
  if (!dir.exists(outputdir)) {
    dir.create(outputdir, recursive = TRUE)
  }
  
  # 保存网络图
  pdf_file <- file.path(outputdir, paste0("network_2x2_", comp_name, ".pdf"))
  pdf(pdf_file, width = 12, height = 10)
  
  par(mar = c(0, 0, 2, 0), cex.main = 1.0)
  
  plot(g,
       layout = layout_with_fr(g),
       vertex.label = V(g)$name,
       main = paste("Cell-Cell Communication Network (2×2 design):", comp_name),
       sub = paste("基于一致性评分 |", nrow(network_data), "个交互作用"))
  
  # 添加图例
  legend("bottomleft",
         legend = c("A > B (上调)", "A < B (下调)", 
                   "高优先级(4/4一致)", "中高优先级(3/4)", "中优先级(2/4)"),
         col = c("#E76F51", "#2A9D8F", "black", "black", "black"),
         lwd = c(3, 3, 4, 3, 2),
         lty = 1,
         bty = "n",
         cex = 0.7)
  
  # 添加样本信息
  legend("bottomright",
         legend = c("注意：每组只有2个样本", "统计功效有限", "结果需谨慎解释"),
         bty = "n",
         cex = 0.6,
         text.col = "red")
  
  dev.off()
  cat("网络图已保存至:", pdf_file, "\n")
  
  # 保存详细结果
  csv_file <- file.path(outputdir, paste0("statistics_2x2_", comp_name, ".csv"))
  write.csv(statistical_results, csv_file, row.names = FALSE)
  
  return(list(
    graph = g,
    network_data = network_data,
    statistical_results = statistical_results
  ))
}

# ============================================================================
# 5. 环形图可视化函数
# ============================================================================

create_directional_circos <- function(statistical_results, comp_name, outputdir, n_top = 20) {
  
  # 筛选top交互
   top_data <- statistical_results %>%
      filter(p_value_raw < 0.05) %>%  # 首先筛选显著的结果
      arrange(desc(abs(mean_diff))) %>%  # 按效应大小排序
      filter(n_up>3 | n_down >3 ) %>%
      head(n_top) 
  
  if (nrow(top_data) == 0) {
    cat("没有足够的数据可绘制环形图\n")
    return(NULL)
  }
  
  # 获取所有细胞类型
  all_cells <- unique(c(top_data$source, top_data$target))
  
  # 创建PDF
  circos_file <- file.path(outputdir, paste0("directional_circos_", comp_name, ".pdf"))
  pdf(circos_file, width = 12, height = 12)
  
  # 初始化circos
  circos.clear()
  circos.par("gap.degree" = 3, "start.degree" = 90, 
             "track.height" = 0.08, "cell.padding" = c(0, 0, 0, 0))
  
  # 初始化扇形
  circos.initialize(factors = factor(all_cells, levels = all_cells), 
                    xlim = c(0, 1))
  
  # 绘制细胞类型轨道
  circos.trackPlotRegion(
    ylim = c(0, 1),
    track.height = 0.1,
    bg.border = "gray70",
    bg.col = "white",
    panel.fun = function(x, y) {
      sector_index <- CELL_META$sector.index
      circos.text(
        x = 0.5,
        y = 0.5,
        labels = sector_index,
        sector.index = sector_index,
        facing = "clockwise",
        niceFacing = TRUE,
        adj = c(0.5, 0.5),
        cex = 0.9,
        col = "black",
        font = 2
      )
    }
  )
  
  # 绘制带箭头的连线
  for (i in 1:nrow(top_data)) {
    source_cell <- top_data$source[i]
    target_cell <- top_data$target[i]
    mean_diff <- top_data$mean_diff[i]
    
    # 设置颜色和宽度
    color <- ifelse(mean_diff > 0, "#E76F51", "#2A9D8F")
    width <- max(1, min(abs(mean_diff) * 100, 8))
    
    # 计算串联位置
    source_y <- 0.7 + (0.2 * (i-1) / nrow(top_data))
    target_y <- 0.1 + (0.2 * (i-1) / nrow(top_data))
    
    # 绘制带箭头的连线
    circos.link(
      sector.index1 = source_cell,
      point1 = source_y,
      sector.index2 = target_cell,
      point2 = target_y,
      col = color,
      lwd = width,
      border = NA,
      arr.type = "triangle",
      arr.length = 1,
      arr.width = 1,
      arr.lwd = 1,
      arr.col = color,
      directional = 1
    )
  }
  
  # 添加图例和标题
  par(new = TRUE)
  plot(0, 0, type = "n", xlim = c(0, 1), ylim = c(0, 1), 
       axes = FALSE, xlab = "", ylab = "")
  
  legend("bottomleft",
         legend = c("Source → Target", 
                   "Salmon: A > B (上调)", 
                   "Olive: A < B (下调)"),
         col = c("black", "#E76F51", "#2A9D8F"),
         lty = c(NA, 1, 1),
         lwd = c(NA, 3, 3),
         pch = c(NA, NA, NA),
         bty = "n", 
         cex = 0.8)
  
  title(main = paste("Directed Cell Communication Network:", comp_name),
        sub = paste("Top", n_top, "interactions | Arrows show direction"),
        cex.main = 1.2, cex.sub = 0.8)
  
  circos.clear()
  dev.off()
  
  cat("带箭头的环形图已保存至:", circos_file, "\n")
  return(top_data)
}

# ============================================================================
# 6. 保守可视化函数
# ============================================================================

generate_conservative_plots <- function(statistical_results, comp_name, outputdir) {
  
  # 创建输出目录
  if (!dir.exists(outputdir)) {
    dir.create(outputdir, recursive = TRUE)
  }
  
  # 1. 一致性分析图
  p1 <- ggplot(statistical_results, aes(x = consistency_score, y = abs(mean_diff))) +
    geom_point(aes(color = priority_level, 
                   size = ifelse(is.na(p_adjusted), 1, -log10(p_adjusted + 1e-10))), 
               alpha = 0.7) +
    scale_color_manual(values = c("high" = "#E76F51", 
                                 "medium_high" = "#F4A261", 
                                 "medium" = "#2A9D8F", 
                                 "low" = "gray")) +
    labs(title = paste("Consistency vs Effect Size:", comp_name),
         x = "Consistency Score (1-4)",
         y = "Absolute Mean Difference",
         color = "Priority Level",
         size = "-log10(p)") +
    theme_minimal()
  
  ggsave(file.path(outputdir, paste0("consistency_plot_", comp_name, ".pdf")), 
         p1, width = 10, height = 8)
  
  # 2. 方向一致性分布
  dir_data <- statistical_results %>%
    count(dominant_direction) %>%
    mutate(prop = n / sum(n),
           dominant_direction = factor(dominant_direction,
                                       levels = c("all_up", "mostly_up", 
                                                 "inconsistent", 
                                                 "mostly_down", "all_down")))
  
  p2 <- ggplot(dir_data, aes(x = dominant_direction, y = prop, 
                             fill = dominant_direction)) +
    geom_col() +
    geom_text(aes(label = paste0(n, "\n(", round(prop*100, 1), "%)")), 
              vjust = -0.3, size = 3) +
    scale_fill_manual(values = c("all_up" = "#E76F51", 
                                "mostly_up" = "#F4A261",
                                "inconsistent" = "gray70",
                                "mostly_down" = "#8AB17D", 
                                "all_down" = "#2A9D8F")) +
    labs(title = paste("Direction Consistency Distribution:", comp_name),
         x = "Dominant Direction",
         y = "Proportion") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
  
  ggsave(file.path(outputdir, paste0("direction_distribution_", comp_name, ".pdf")), 
         p2, width = 8, height = 6)
  
  # 3. Top 20 交互条形图
   top_data <- statistical_results %>%
      filter(p_value_raw < 0.05) %>%  # 首先筛选显著的结果
      arrange(desc(abs(mean_diff))) %>%  # 按效应大小排序
      filter(n_up>3 | n_down >3 ) %>%
      head(20) 
  
  if (nrow(top_data) > 0) {
    top_data$interaction <- paste(top_data$source, "→", top_data$target)
    top_data$interaction <- factor(top_data$interaction, 
                                  levels = top_data$interaction[order(top_data$mean_diff)])
    
    p3 <- ggplot(top_data, aes(x = interaction, y = mean_diff)) +
      geom_bar(aes(fill = ifelse(mean_diff > 0, "#E76F51", "#2A9D8F")), 
               stat = "identity", width = 0.7) +
      scale_fill_identity() +
      coord_flip() +
      labs(title = paste("Top 20 Largest Changes:", comp_name),
           x = "Cell-Cell Interaction",
           y = "Mean Difference (A - B)") +
      theme_minimal() +
      theme(axis.text.y = element_text(size = 9))
    
    ggsave(file.path(outputdir, paste0("top20_changes_", comp_name, ".pdf")), 
           p3, width = 10, height = 8)
  }
  
  # 4. 生成环形图
  circos_data <- create_directional_circos(statistical_results, comp_name, outputdir, n_top = 20)
  
  cat("\n所有可视化已保存至:", outputdir, "\n")
  cat("生成的图形：\n")
  cat("1. consistency_plot_", comp_name, ".pdf\n", sep = "")
  cat("2. direction_distribution_", comp_name, ".pdf\n", sep = "")
  cat("3. top20_changes_", comp_name, ".pdf\n", sep = "")
  cat("4. directional_circos_", comp_name, ".pdf\n", sep = "")
  
  return(list(
    consistency_plot = p1,
    direction_plot = p2,
    top_changes_plot = if (exists("p3")) p3 else NULL,
    circos_data = circos_data
  ))
}

# ============================================================================
# 7. 完整工作流程函数
# ============================================================================

run_complete_analysis_2x2 <- function(cellchat_objects, sample_names, 
                                      group_A, group_B, target_types, 
                                      comp_name, outputdir) {
  
  # 打印标题
  separator_line <- paste(rep("=", 60), collapse = "")
  
  cat(separator_line, "\n")
  cat("开始2×2分析:", comp_name, "\n")
  cat("Group A:", paste(group_A, collapse = ", "), "\n")
  cat("Group B:", paste(group_B, collapse = ", "), "\n")
  cat("目标细胞类型:", length(target_types), "种\n")
  cat(separator_line, "\n\n")
  
  # 步骤1: 提取通讯矩阵
  cat("步骤1: 提取通讯矩阵...\n")
  all_matrices <- list()
  
  for (sample_name in sample_names) {
    cat("处理样本:", sample_name, "...")
    
    if (!sample_name %in% names(cellchat_objects)) {
      cat("错误: 未找到样本", sample_name, "的CellChat对象\n")
      next
    }
    
    matrix_df <- extract_communication_matrix(
      cellchat_objects[[sample_name]], 
      sample_name, 
      target_types
    )
    
    all_matrices[[sample_name]] <- matrix_df
    cat("完成\n")
  }
  
  combined_data <- bind_rows(all_matrices)
  
  # 步骤2: 执行所有配对比较
  cat("\n步骤2: 执行配对比较...\n")
  all_comparisons <- list()
  pair_counter <- 1
  
  for (a_sample in group_A) {
    for (b_sample in group_B) {
      cat("比较:", a_sample, "vs", b_sample, "...")
      
      comparison_result <- compare_two_samples(combined_data, a_sample, b_sample)
      comparison_result$pair_id <- pair_counter
      comparison_result$comparison_name <- paste(a_sample, "vs", b_sample)
      
      all_comparisons[[as.character(pair_counter)]] <- comparison_result
      pair_counter <- pair_counter + 1
      
      cat("完成\n")
    }
  }
  
  all_pairwise_results <- bind_rows(all_comparisons)
  
  # 步骤3: 统计分析
  cat("\n步骤3: 统计分析...\n")
  statistical_results <- perform_statistical_analysis_2x2(
    all_pairwise_results, 
    group_A, 
    group_B
  )
  
  # 步骤4: 网络可视化
  cat("\n步骤4: 网络可视化...\n")
  network_result <- plot_cell_network_2x2(statistical_results, comp_name, outputdir)
  
  # 步骤5: 生成保守可视化
  cat("\n步骤5: 生成可视化...\n")
  visualization_results <- generate_conservative_plots(
    statistical_results, 
    comp_name, 
    outputdir
  )
  
  cat("\n", separator_line, "\n")
  cat("分析完成! 结果已保存至:", outputdir, "\n")
  cat("注意: 2×2设计统计功效有限，请谨慎解释结果\n")
  cat(separator_line, "\n")
  
  # 返回所有结果
  return(list(
    combined_data = combined_data,
    all_comparisons = all_pairwise_results,
    statistical_results = statistical_results,
    network_result = network_result,
    visualization_results = visualization_results,
    summary = list(
      total_interactions = nrow(statistical_results),
      high_consistency = sum(statistical_results$consistency_score == 4),
      significant_05 = sum(statistical_results$p_value_raw < 0.05, na.rm = TRUE),
      significant_10 = sum(statistical_results$p_value_raw < 0.1, na.rm = TRUE)
    )
  ))
}


# ============================================================================
# 8. 批量分析多个比较
# ============================================================================

run_batch_analyses <- function(cellchat_objects, comparisons_list, target_types, output_base_dir) {
  results <- list()
  
  for (i in seq_along(comparisons_list)) {
    comp <- comparisons_list[[i]]
    comp_name <- comp$name
    group_A <- comp$group_A
    group_B <- comp$group_B
    
    cat("\n开始分析:", comp_name, "\n")
    cat("Group A:", paste(group_A, collapse = ", "), "\n")
    cat("Group B:", paste(group_B, collapse = ", "), "\n")
    
    # 创建输出目录
    outputdir <- file.path(output_base_dir, comp_name)
    
    # 运行分析
    result <- tryCatch({
      run_complete_analysis_2x2(
        cellchat_objects = cellchat_objects,
        sample_names = unique(c(group_A, group_B)),
        group_A = group_A,
        group_B = group_B,
        target_types = target_types,
        comp_name = comp_name,
        outputdir = outputdir
      )
    }, error = function(e) {
      cat("分析失败:", e$message, "\n")
      return(NULL)
    })
    
    if (!is.null(result)) {
      results[[comp_name]] <- result
    }
  }
  
  return(results)
}

# 打印加载信息
cat("CellChat 2×2 设计分析工具已加载\n")
cat("版本: 2.0 (修复版)\n")
cat("主要功能:\n")
cat("1. 2×2设计的细胞通讯分析\n")
cat("2. 小样本统计分析\n")
cat("3. 多种可视化输出\n")
cat("4. 完整工作流程\n\n")
cat("输入 example_usage() 查看使用示例\n")
cat("输入 check_data_compatibility() 检查数据兼容性\n")

CellChat 2×2 设计分析工具已加载
版本: 2.0 (修复版)
主要功能:
1. 2×2设计的细胞通讯分析
2. 小样本统计分析
3. 多种可视化输出
4. 完整工作流程

输入 example_usage() 查看使用示例
输入 check_data_compatibility() 检查数据兼容性


In [219]:
target_celltypes <- c(#'CD4.exh', 'CD4.naive', 'CD4.reg',
                      'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif')

In [384]:
# 运行分析
comp_name =  "oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro"
results <- run_complete_analysis_2x2(
  cellchat_objects = cellchats,  # 您的CellChat对象列表
  sample_names =  c('M20_5_V21','M20_5_V47','M20_3_V4','M20_3_V5'),
  group_A = c('M20_5_V21','M20_5_V47'),  # 对照组，2个样本
  group_B = c('M20_3_V4','M20_3_V5'),  # 处理组，2个样本
 target_types = c('CD4.exh', 'CD4.naive', 'CD4.reg',
                      #'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                    'TAM.plac8', 'TAM.prolif'),
  comp_name = comp_name,
  outputdir = paste0(outputdir,comp_name)
)



开始2×2分析: oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro 
Group A: M20_5_V21, M20_5_V47 
Group B: M20_3_V4, M20_3_V5 
目标细胞类型: 9 种

步骤1: 提取通讯矩阵...
处理样本: M20_5_V21 ...样本 M20_5_V21 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_5_V47 ...样本 M20_5_V47 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_3_V4 ...样本 M20_3_V4 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_3_V5 ...样本 M20_3_V5 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成

步骤2: 执行配对比较...
比较: M20_5_V21 vs M20_3_V4 ...完成
比较: M20_5_V21 vs M20_3_V5 ...完成
比较: M20_5_V47 vs M20_3_V4 ...完成
比较: M20_5_V47 vs M20_3_V5 ...完成

步骤3: 统计分析...
统计设计: 每组2个样本，共有 4 个比较对
注意: 统计功效有限，建议谨慎解释结果

统计分析摘要（2×2设计）:
总交互作用数量: 81 
高度一致(4/4同向): 7 
中等一致(3/4同向): 6 
边缘一致(2/4同向): 17

Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → CD4.naive' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → CD4.naive' in 'mbcsToSbcs': dot substituted for <86>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → CD4.naive' in 'mbcsToSbcs': dot substituted for <92>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → TAM.Ccr2' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → TAM.Ccr2' in 'mbcsToSbcs': dot substituted for <86>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → TAM.Ccr2' in 'mbcsToSbcs': dot substituted

带箭头的环形图已保存至: transcriptome/pool_DEG/oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro/directional_circos_oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro.pdf 

所有可视化已保存至: transcriptome/pool_DEG/oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro 
生成的图形：
1. consistency_plot_oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro.pdf
2. direction_distribution_oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro.pdf
3. top20_changes_oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro.pdf
4. directional_circos_oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro.pdf

分析完成! 结果已保存至: transcriptome/pool_DEG/oAd-exCD40L_vs_oAd-wtCD40L_2x2_CD42Macro 
注意: 2×2设计统计功效有限，请谨慎解释结果


In [385]:
# 2. 加载你的CellChat对象
# cellchat_objects <- list(...)
comp_name =  "oAd-exCD40L_vs_oAd_2x2_CD42Macro"
results <- run_complete_analysis_2x2(
  cellchat_objects = cellchats,  # 您的CellChat对象列表
  sample_names =  c('M20_5_V21','M20_5_V47','M20_V40','M20_V42'),# put test group at tail
  group_A = c('M20_5_V21','M20_5_V47'),  # 对照组，2个样本
  group_B = c('M20_V40','M20_V42'),  # 处理组，2个样本
 target_types = c('CD4.exh', 'CD4.naive', 'CD4.reg',
                      #'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif'),
  comp_name = comp_name,
  outputdir = paste0(outputdir,comp_name)
)



开始2×2分析: oAd-exCD40L_vs_oAd_2x2_CD42Macro 
Group A: M20_5_V21, M20_5_V47 
Group B: M20_V40, M20_V42 
目标细胞类型: 9 种

步骤1: 提取通讯矩阵...
处理样本: M20_5_V21 ...样本 M20_5_V21 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_5_V47 ...样本 M20_5_V47 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_V40 ...样本 M20_V40 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_V42 ...样本 M20_V42 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成

步骤2: 执行配对比较...
比较: M20_5_V21 vs M20_V40 ...完成
比较: M20_5_V21 vs M20_V42 ...完成
比较: M20_5_V47 vs M20_V40 ...完成
比较: M20_5_V47 vs M20_V42 ...完成

步骤3: 统计分析...
统计设计: 每组2个样本，共有 4 个比较对
注意: 统计功效有限，建议谨慎解释结果

统计分析摘要（2×2设计）:
总交互作用数量: 81 
高度一致(4/4同向): 10 
中等一致(3/4同向): 2 
边缘一致(2/4同向): 16 
显著(p<0.05): 3 


Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → CD4.naive' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → CD4.naive' in 'mbcsToSbcs': dot substituted for <86>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'TAM.M2 → CD4.naive' in 'mbcsToSbcs': dot substituted for <92>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.naive → TAM.M2' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.naive → TAM.M2' in 'mbcsToSbcs': dot substituted for <86>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.naive → TAM.M2' in 'mbcsToSbcs': dot substitu

带箭头的环形图已保存至: transcriptome/pool_DEG/oAd-exCD40L_vs_oAd_2x2_CD42Macro/directional_circos_oAd-exCD40L_vs_oAd_2x2_CD42Macro.pdf 

所有可视化已保存至: transcriptome/pool_DEG/oAd-exCD40L_vs_oAd_2x2_CD42Macro 
生成的图形：
1. consistency_plot_oAd-exCD40L_vs_oAd_2x2_CD42Macro.pdf
2. direction_distribution_oAd-exCD40L_vs_oAd_2x2_CD42Macro.pdf
3. top20_changes_oAd-exCD40L_vs_oAd_2x2_CD42Macro.pdf
4. directional_circos_oAd-exCD40L_vs_oAd_2x2_CD42Macro.pdf

分析完成! 结果已保存至: transcriptome/pool_DEG/oAd-exCD40L_vs_oAd_2x2_CD42Macro 
注意: 2×2设计统计功效有限，请谨慎解释结果


In [386]:
# 运行分析
comp_name =  "oAd-wtCD40L_vs_oAd_2x2_CD42Macro"
results <- run_complete_analysis_2x2(
  cellchat_objects = cellchats,  # 您的CellChat对象列表
  sample_names =  c('M20_3_V4','M20_3_V5','M20_V40','M20_V42'),# put test group at tail
  group_A = c('M20_3_V4','M20_3_V5'),  # 对照组，2个样本
  group_B = c('M20_V40','M20_V42'),  # 处理组，2个样本
 target_types = c('CD4.exh', 'CD4.naive', 'CD4.reg',
                     # 'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif'),
  comp_name = comp_name,
  outputdir = paste0(outputdir,comp_name)
)



开始2×2分析: oAd-wtCD40L_vs_oAd_2x2_CD42Macro 
Group A: M20_3_V4, M20_3_V5 
Group B: M20_V40, M20_V42 
目标细胞类型: 9 种

步骤1: 提取通讯矩阵...
处理样本: M20_3_V4 ...样本 M20_3_V4 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_3_V5 ...样本 M20_3_V5 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_V40 ...样本 M20_V40 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成
处理样本: M20_V42 ...样本 M20_V42 中找到的细胞亚群: 9 / 9 
具体亚群: CD4.exh, CD4.naive, CD4.reg, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 
完成

步骤2: 执行配对比较...
比较: M20_3_V4 vs M20_V40 ...完成
比较: M20_3_V4 vs M20_V42 ...完成
比较: M20_3_V5 vs M20_V40 ...完成
比较: M20_3_V5 vs M20_V42 ...完成

步骤3: 统计分析...
统计设计: 每组2个样本，共有 4 个比较对
注意: 统计功效有限，建议谨慎解释结果

统计分析摘要（2×2设计）:
总交互作用数量: 81 
高度一致(4/4同向): 11 
中等一致(3/4同向): 2 
边缘一致(2/4同向): 3 
显著(p<0.05): 3 
边缘显著(p<0.1)

Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.exh → CD4.exh' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.exh → CD4.exh' in 'mbcsToSbcs': dot substituted for <86>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.exh → CD4.exh' in 'mbcsToSbcs': dot substituted for <92>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.exh → CD4.naive' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.exh → CD4.naive' in 'mbcsToSbcs': dot substituted for <86>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD4.exh → CD4.naive' in 'mbcsToSbcs': dot substitu

带箭头的环形图已保存至: transcriptome/pool_DEG/oAd-wtCD40L_vs_oAd_2x2_CD42Macro/directional_circos_oAd-wtCD40L_vs_oAd_2x2_CD42Macro.pdf 

所有可视化已保存至: transcriptome/pool_DEG/oAd-wtCD40L_vs_oAd_2x2_CD42Macro 
生成的图形：
1. consistency_plot_oAd-wtCD40L_vs_oAd_2x2_CD42Macro.pdf
2. direction_distribution_oAd-wtCD40L_vs_oAd_2x2_CD42Macro.pdf
3. top20_changes_oAd-wtCD40L_vs_oAd_2x2_CD42Macro.pdf
4. directional_circos_oAd-wtCD40L_vs_oAd_2x2_CD42Macro.pdf

分析完成! 结果已保存至: transcriptome/pool_DEG/oAd-wtCD40L_vs_oAd_2x2_CD42Macro 
注意: 2×2设计统计功效有限，请谨慎解释结果


In [155]:
# 细胞亚群互作网络图
plot_cell_network <- function(lr_result, comp_name) {
  # 汇总每个细胞对的总体变化
  cell_pair_summary <- lr_result %>%
    group_by(source, target) %>%
    summarise(
      total_change = sum(abs(difference)),
      mean_change = mean(difference),
      n_lr_pairs = n(),
      .groups = "drop"
    )
  
  # 创建网络图
  library(igraph)
  
  # 创建图对象
  g <- graph_from_data_frame(cell_pair_summary, directed = TRUE)
  
  # 设置边的属性
  E(g)$weight <- cell_pair_summary$total_change
  E(g)$color <- ifelse(cell_pair_summary$mean_change > 0, "purple", "seagreen")
  E(g)$width <- scales::rescale(cell_pair_summary$total_change, to = c(1, 8))
  
  # 设置节点的属性（细胞类型）
  V(g)$size <- 15
  V(g)$label.cex <- 0.8
  
  # 绘制网络图
  pdf(paste0(outputdir,"network_", comp_name, ".pdf"), width = 12, height = 10)
  plot(g, 
       layout = layout_with_fr(g),
       edge.arrow.size = 0.5,
       edge.arrow.width = 1,
       main = paste("network:", comp_name))
  legend("bottomleft", 
         legend = c("up", "dn"), 
         col = c("purple", "seagreen"), 
         lwd = 2, 
         bty = "n")
  dev.off()
}

In [240]:
outputdir=paste0("transcriptome/pool_DEG/","CD8_Macro_cells_comp/")

In [241]:
# 2. 定义要分析的细胞亚群
target_celltypes <- c(#'CD4.exh', 'CD4.naive', 'CD4.reg',
                      'CD8.cyto', 'CD8.naive', 'CD8.prolif', 'CD8.RM',
                      #'ID8', 'ID8_ADV', 'ID8_apop', 'ID8_apop_ADV', 'ID8_prolif', 'ID8_prolif_ADV', 
                      'TAM.Ccl', 'TAM.Ccr2', 'TAM.Fth1', 'TAM.M2',
                     'TAM.plac8', 'TAM.prolif')

In [175]:


dir.create(outputdir, showWarnings = TRUE, recursive = FALSE)
# 4. 为所有样本提取communication数据
all_comm_data <- list()
for (i in 1:length(cellchats)) {
  sample_name <- names(cellchats)[i]
  cat("\n正在处理样本:", sample_name, "\n")
  all_comm_data[[sample_name]] <- extract_communication_matrix(cellchats[[i]], sample_name, target_celltypes)
}

# 合并所有数据
comm_df <- bind_rows(all_comm_data)
# 7. 执行所有样本对的比较
comparison_results <- list()

for (pair in sample_pairs) {
  sample1 <- pair[1]
  sample2 <- pair[2]
  
  cat("\n==================================================\n")
  cat("比较:", sample1, "vs", sample2, "\n")
  cat("==================================================\n")
  
  result <- compare_two_samples(comm_df, sample1, sample2)
  comparison_name <- paste0(sample1, "_vs_", sample2)
  comparison_results[[comparison_name]] <- result
  
  # 统计信息
  n_up <- sum(result$change_direction == "upregulated")
  n_down <- sum(result$change_direction == "downregulated")
  n_total <- nrow(result)
  
  cat("总共有", n_total, "个细胞亚群对在两个样本中都存在\n")
  cat("上调的communication:", n_up, "\n")
  cat("下调的communication:", n_down, "\n")
  cat("不变的communication:", n_total - n_up - n_down, "\n")
  
  # 保存结果
  write.csv(result, paste0(outputdir,"communication_changes_", comparison_name, ".csv"), row.names = FALSE)
}

# 8. 生成汇总报告
cat("\n\n最终汇总报告:\n")
cat("==================================================\n")

for (comp_name in names(comparison_results)) {
  result <- comparison_results[[comp_name]]
  
  # 找出变化最显著的interaction
  top_up <- result %>% 
    filter(change_direction == "upregulated") %>% 
    arrange(desc(strength_diff)) %>% 
    head(5)
  
  top_down <- result %>% 
    filter(change_direction == "downregulated") %>% 
    arrange(strength_diff) %>% 
    head(5)
  
  cat("\n比较:", comp_name, "\n")
  
  if (nrow(top_up) > 0) {
    cat("上调最显著的interactions:\n")
    for (i in 1:nrow(top_up)) {
      cat("  ", top_up$source[i], "->", top_up$target[i], 
          "(差异:", round(top_up$strength_diff[i], 4), ")\n")
    }
  }
  
  if (nrow(top_down) > 0) {
    cat("下调最显著的interactions:\n")
    for (i in 1:nrow(top_down)) {
      cat("  ", top_down$source[i], "->", top_down$target[i], 
          "(差异:", round(top_down$strength_diff[i], 4), ")\n")
    }
  }
  cat("---\n")
}



正在处理样本: M20_3_V4 
样本 M20_3_V4 中找到的细胞亚群: 10 / 10 
具体亚群: CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 

正在处理样本: M20_3_V5 
样本 M20_3_V5 中找到的细胞亚群: 10 / 10 
具体亚群: CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 

正在处理样本: M20_5_V21 
样本 M20_5_V21 中找到的细胞亚群: 10 / 10 
具体亚群: CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 

正在处理样本: M20_5_V47 
样本 M20_5_V47 中找到的细胞亚群: 10 / 10 
具体亚群: CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 

正在处理样本: M20_V40 
样本 M20_V40 中找到的细胞亚群: 10 / 10 
具体亚群: CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 

正在处理样本: M20_V42 
样本 M20_V42 中找到的细胞亚群: 10 / 10 
具体亚群: CD8.cyto, CD8.naive, CD8.prolif, CD8.RM, TAM.Ccl, TAM.Ccr2, TAM.Fth1, TAM.M2, TAM.plac8, TAM.prolif 

比较: M20_3_V4 vs M20_3_V5 
总共有 100 个细胞亚群对在两个样本中都存在
上调的comm

In [176]:

# 为每个比较生成图表
for (comp_name in names(comparison_results)) {
  p <- generate_comparison_plot(comparison_results[[comp_name]], comp_name)
  if (!is.null(p)) {
    ggsave(paste0(outputdir,"visualization_", comp_name, "_CD4_Macro.pdf"), p, width = 10, height = 8)
    cat("已生成可视化图表:", paste0(outputdir,"visualization_", comp_name, ".pdf"), "\n")
  }
}

cat("\n分析完成！所有结果已保存到CSV文件和PDF图表中。\n")

已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_3_V4_vs_M20_3_V5.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_5_V21_vs_M20_3_V4.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_5_V47_vs_M20_3_V4.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_3_V4_vs_M20_V40.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_3_V4_vs_M20_V42.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_5_V21_vs_M20_3_V5.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_5_V47_vs_M20_3_V5.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_3_V5_vs_M20_V40.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_3_V5_vs_M20_V42.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visualization_M20_5_V21_vs_M20_5_V47.pdf 
已生成可视化图表: transcriptome/pool_DEG/CD8_Macro_cells_comp/visu